# 1. Import Libraries

In [1]:
import pandas as pd
import re
import json
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Load Dataset

In [2]:
path = r"C:\Users\shaik\OneDrive\Desktop\feedbackanalysissystem\data\Education_Feedback.csv"

df = pd.read_csv(path)

# Drop unwanted column safely
df = df.drop(columns=["Unnamed: 0"], errors='ignore')

# Keep required columns
df = df[["StudentComments", "Sentiment"]]

# Remove null values
df = df.dropna()

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (2007747, 2)


,StudentComments,Sentiment
0,good,positive
1,good,positive
2,teacher,positive
3,friendly teacher but not enough ability to enc...,positive
4,teacher,positive


# 3. Text Cleaning

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df["clean_text"] = df["StudentComments"].apply(clean_text)

df.head()

,StudentComments,Sentiment,clean_text
0,good,positive,good
1,good,positive,good
2,teacher,positive,teacher
3,friendly teacher but not enough ability to enc...,positive,friendly teacher but not enough ability to enc...
4,teacher,positive,teacher


# 4. Word Lists

In [4]:
positive_words = [
    "good","excellent","nice","friendly","helpful","amazing",
    "great","supportive","clear","awesome","love","best"
]

negative_words = [
    "bad","poor","worst","rude","confusing","boring",
    "slow","unclear","hate","terrible","problem"
]

negations = ["not", "no", "never", "hardly"]

intensifiers = ["very","extremely","really","highly","too","so","super"]

diminishers = ["slightly","somewhat","little","barely"]


contrast_words = ["but", "however", "although", "though"]

strong_positive = ["excellent", "amazing", "outstanding", "fantastic"]
strong_negative = ["worst", "terrible", "useless", "horrible"]

domain_positive = ["easy", "understandable", "clear", "interactive"]
domain_negative = ["confusing", "fast", "lengthy", "complex", "unclear"]

sarcasm_phrases = ["yeah right", "as if", "just great", "wow"]

# 5. Phrase-Level Rules

In [5]:
phrase_rules = {
    "not good": "negative",
    "not helpful": "negative",
    "not clear": "negative",
    "very good": "positive",
    "too fast": "negative",
    "too confusing": "negative",
    "easy to understand": "positive",
    "well explained": "positive"
}

def check_phrases(text):
    for phrase, sentiment in phrase_rules.items():
        if phrase in text:
            return sentiment
    return None

# 6. Sentiment Function

In [6]:
def advanced_sentiment(text):

    text = text.lower()

    #  Phrase check 
    phrase_result = check_phrases(text)
    if phrase_result:
        return phrase_result

    #  Sarcasm detection
    for phrase in sarcasm_phrases:
        if phrase in text:
            return "negative"

    words = text.split()
    score = 0

    #  Handle contrast words 
    for cw in contrast_words:
        if cw in words:
            idx = words.index(cw)
            words = words[idx+1:]
            break

    for i, word in enumerate(words):

        window = words[max(0, i-3):i]

        negated = any(w in negations for w in window)
        intensified = any(w in intensifiers for w in window)
        diminished = any(w in diminishers for w in window)

        weight = 1

        if intensified:
            weight = 2
        elif diminished:
            weight = 0.5

        #  Strong words
        if word in strong_positive:
            score += 3
            continue

        if word in strong_negative:
            score -= 3
            continue

        #  Domain words
        if word in domain_positive:
            score += 1.5
            continue

        if word in domain_negative:
            score -= 1.5
            continue

        # Normal words
        if word in positive_words:
            if negated:
                score -= weight
            else:
                score += weight

        elif word in negative_words:
            if negated:
                score += weight
            else:
                score -= weight

    # Final classification
    if score > 0.5:
        return "positive"
    elif score < -0.5:
        return "negative"
    else:
        return "neutral"

# 7. Apply Model

In [7]:
df["prediction"] = df["clean_text"].apply(advanced_sentiment)

df[["StudentComments","Sentiment","prediction"]].head()

,StudentComments,Sentiment,prediction
0,good,positive,positive
1,good,positive,positive
2,teacher,positive,neutral
3,friendly teacher but not enough ability to enc...,positive,neutral
4,teacher,positive,neutral


# 8. Evaluation

In [8]:
y_true = df["Sentiment"]
y_pred = df["prediction"]

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.7411093130758009

Classification Report:

              precision    recall  f1-score   support

    negative       0.66      0.26      0.37    141203
     neutral       0.11      0.31      0.16    144505
    positive       0.91      0.82      0.86   1722039

    accuracy                           0.74   2007747
   macro avg       0.56      0.46      0.47   2007747
weighted avg       0.83      0.74      0.78   2007747


Confusion Matrix:

[[  36677   57936   46590]
 [   7254   45238   92013]
 [  11907  304087 1406045]]


# 10. Save Model

In [9]:
model = {
    "positive_words": positive_words,
    "negative_words": negative_words,
    "negations": negations,
    "intensifiers": intensifiers,
    "diminishers": diminishers,
    "contrast_words": contrast_words,
    "strong_positive": strong_positive,
    "strong_negative": strong_negative,
    "domain_positive": domain_positive,
    "domain_negative": domain_negative,
    "sarcasm_phrases": sarcasm_phrases,
    "phrase_rules": phrase_rules
}

with open("rule_model.json", "w") as f:
    json.dump(model, f)

print("Rule-Based Model Saved Successfully")

Rule-Based Model Saved Successfully
